# Lab 3: Object Oriented Programming Part I

Welcome to Lab 3 of Programming Methods! In this class we will start exploring Object Oriented Programming (OOP). The Lab has **three** main exercises and **six** wrap-up questions in the end. There is one "optional stretch" exercise.

Create a **new branch** for this Lab (e.g.,`git checkout -b lab-03-oop1`).

Then, complete the following exercises in **this** Jupyter Notebook. 

Once you are done **commit your work** using the proper Git commands.

The **solutions** for this lab will be published on the time of the next Lab.

Good luck :) 



### From scripts to reusable API clients

In the last class your code:
- Fetched data from the World Bank REST API
- Sent a prompt to a local Ollama model and parsed the JSON reply
- Built a simple chat loop against `ollama.chat()`

That code has three recurring problems:

1. **No validation** — you trust that the JSON has the shape you expect. If a field is missing or renamed, your code fails.
2. **Repetition** — you check `response.status_code == 200` and call `.json()` in multiple places.
3. **No reuse** — the logic for "format a prompt" and "send a request" is tangled together inside one script, so you can't easily plug in a new API without rewriting everything.

Today we will fix all three using **Pydantic** (validation) and **class hierarchies** (reuse). Last class's code will not be thrown away, it will be *refactored*.

By the end of this notebook you should have:
- The Pydantic model **OllamaResponse**.
- The base class **APIClient** with the shared request logic.
- The subclass **OllamaClient**.
- The class **ChatSession** that wraps the chat loop from Lab 2 Exercise 3.

## Setup

Run this cell first.


In [2]:
import requests
from pydantic import BaseModel, ValidationError

---
## Exercise 1: Classes and Pydantic Models

In the previous lab, we interacted with Ollama directly using `requests`.

In this exercise, we will improve that code in two steps:

1. Encapsulate the interaction with Ollama inside a **class**.
2. Use **Pydantic** to describe and validate the response returned by the API.

### 1a. Create an OllamaClient class

Create a class called `OllamaClient` that represents a client for the local Ollama API.

The class should:

- Receive the model name when instantiated.
- Store the Ollama API URL as an attribute.
- Have a method called `generate(prompt)` that:
    - Receives a prompt as a string.
    - Sends a `POST` request to `http://localhost:11434/api/generate`.
    - Sends the model and prompt in the JSON body.
    - Sets `"stream": False`.
    - Checks that the request was successful.
    - Returns the JSON response as a Python dictionary.

Instantiate the class and use it to ask the model a simple question.

In [80]:
class OllamaClient:
   def __init__(self, model: str, url: str = "http://localhost:11434/api/generate"):
      self.model = model
      self.url = url

   def generate(self, prompt: str) -> dict:
      response = requests.post(
         url = self.url,
         json={"model": self.model, "prompt": prompt, "stream": False})
      response.raise_for_status()
      return response.json()

client = OllamaClient("llama3.2:1b")
raw = client.generate("Why is the sky blue? Answer in one sentence.")
print(raw)

{'model': 'llama3.2:1b', 'created_at': '2026-09-24T15:01:59.0733101Z', 'response': "The sky appears blue because of a phenomenon called Rayleigh scattering, in which shorter (blue) wavelengths of light are scattered more than longer (red) wavelengths by the tiny molecules of gases in the Earth's atmosphere, resulting in the blue color we see during the day.", 'done': True, 'done_reason': 'stop', 'context': [128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 271, 128009, 128006, 882, 128007, 271, 10445, 374, 279, 13180, 6437, 30, 22559, 304, 832, 11914, 13, 128009, 128006, 78191, 128007, 271, 791, 13180, 8111, 6437, 1606, 315, 264, 25885, 2663, 13558, 64069, 72916, 11, 304, 902, 24210, 320, 12481, 8, 93959, 315, 3177, 527, 38067, 810, 1109, 5129, 320, 1171, 8, 93959, 555, 279, 13987, 35715, 315, 45612, 304, 279, 9420, 596, 16975, 11, 13239, 304, 279, 6437, 1933, 584, 1518, 2391, 279, 1938, 13], 'total_duration': 7492496100, 'load_duration': 6882828000, 'prompt

### 1b. OllamaResponse Pydantic model

Currently, `generate()` returns a regular Python dictionary.

Let's use Pydantic to describe the structure of the response.

Look at the JSON returned by Ollama and create a Pydantic model called
`OllamaResponse`.

At minimum, it should validate:

- **model**: Name of the model used.
- **response**: Text generated by the model.
- **done**: Whether generation has finished.

Then use `OllamaResponse` to validate the dictionary returned by your
`OllamaClient`.

Try accessing the generated text using:

`result.response`

instead of:

`result["response"]`

In [81]:
class OllamaResponse(BaseModel):
    model: str
    response: str
    done: bool

#model_validate validates that what I pass to it follows structure of the BaseModel in this case OllamaResponse
result = OllamaResponse.model_validate(raw)
print(result.response)
print(result.done)

The sky appears blue because of a phenomenon called Rayleigh scattering, in which shorter (blue) wavelengths of light are scattered more than longer (red) wavelengths by the tiny molecules of gases in the Earth's atmosphere, resulting in the blue color we see during the day.
True


### 1c. Integrate Pydantic into the class

Finally, modify `OllamaClient.generate()` so that it returns an
`OllamaResponse` object directly instead of a dictionary.

Add the appropriate return type annotation to the method.

In [82]:
# TODO: update OllamaClient.generate() to return an OllamaResponse.
# TODO: validate and inspect the response.
class OllamaClient:
    def __init__(self, model: str, base_url: str = "http://localhost:11434"):
        self.model = model
        self.base_url = base_url

    def generate(self, prompt: str) -> OllamaResponse:
        response = requests.post(
            f"{self.base_url}/api/generate",
            json={"model": self.model, "prompt": prompt, "stream": False},
            timeout=120,
        )
        if response.status_code != 200:
            raise RuntimeError(
                f"Request failed with status {response.status_code}: {response.text}"
            )
        return OllamaResponse.model_validate(response.json())

client = OllamaClient("llama3.2:1b")
result = client.generate("Why is the sky blue? Answer in one sentence.")
print(type(result))
print(result)

<class '__main__.OllamaResponse'>
model='llama3.2:1b' response='The sky appears blue because of a phenomenon called Rayleigh scattering, in which shorter, blue wavelengths of light are scattered more than longer, red wavelengths by the tiny molecules of gases in the atmosphere, resulting in the blue color we see during the day.' done=True


### 1d. Break it on purpose

1. Pass in a dictionary that is missing a required field, or that has the wrong type for **done**. Wrap the call in a `ValidationError` and print `e.errors()`.
2. Pass in a dictionary that has an additional field. Does it raise an exception?

**Question to answer in a markdown cell below your code:** what does Pydantic's
error message tell you that a plain `KeyError` would not?

In [83]:
# Missing field --> raises
try:
    bad = OllamaResponse(model="llama3.2:1b", response="hi")  # missing `done`
except ValidationError as e:
    print(e.errors())

[{'type': 'missing', 'loc': ('done',), 'msg': 'Field required', 'input': {'model': 'llama3.2:1b', 'response': 'hi'}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [84]:
# Wrong type for `done` --> raises
try:
    bad = OllamaResponse(model="llama3.2:1b", response="hi", done="not a bool")
except ValidationError as e:
    print(e.errors())

[{'type': 'bool_parsing', 'loc': ('done',), 'msg': 'Input should be a valid boolean, unable to interpret input', 'input': 'not a bool', 'url': 'https://errors.pydantic.dev/2.13/v/bool_parsing'}]


In [85]:
# Extra, unexpected field --> does NOT raise, just gets silently ignored
extra_field_response = OllamaResponse(
    model="llama3.2:1b",
    response="hi",
    done=True,
    context=[1, 2, 3],
    prompt_eval_count=17,
)
print(extra_field_response)


model='llama3.2:1b' response='hi' done=True


**Your answer:**

A plain `KeyError` only tells you that *some key* wasn't found (`KeyError: 'done'`), and only the first one. Pydantic's `ValidationError` tells you much more:

- **Every** problem at once, not just the first (`e.errors()` returns a list).
- **Where** it happened (`loc`, e.g. `('done',)`, which also works for nested fields).
- **What kind** of problem it is (`type`: `missing`, `bool_parsing`, ...), so "field absent" and "field present with the wrong type" are distinguishable. A `KeyError` can never detect a wrong type; the bad value would just flow on into the rest of the program.
- A human-readable `msg` and the offending `input` value.

Because validation happens at the boundary, the failure is reported where the bad data *entered*, not several functions later.

On the second experiment: an extra field does **not** raise. By default Pydantic uses `extra="ignore"`, so unknown fields (`context`, `prompt_eval_count`, ...) are silently dropped. That's usually what we want for a third-party API that may add fields, but it can be made strict with `model_config = ConfigDict(extra="forbid")`.

---
## Exercise 2: Base `APIClient` Class Hierarchy

Our `OllamaClient` already handles several responsibilities that are common when interacting with APIs:

- Store a `base_url`
- Send HTTP requests
- Check whether a request was successful
- Parse the JSON response

Instead of implementing these behaviors directly in `OllamaClient`, let's move the common functionality into a reusable **base class** called `APIClient`.

`OllamaClient` will then **inherit** from `APIClient` and add only the behavior that is specific to Ollama.

This will give us the following class hierarchy:

```text
APIClient
    │
    └── OllamaClient
```

The goal is to separate **general API functionality** from **Ollama-specific functionality**, while introducing inheritance and code reuse.

---
## Exercise 3: Handling Validation Errors

Our `OllamaClient.generate()` method now converts the API response into an
`OllamaResponse` Pydantic object.

The cell in Exercise 3a intentionally raises a validation error. Run it by
itself rather than using **Run All**, then continue to Exercise 3b.

But what happens if the API returns data that does **not** match the structure we expect?

### 3a. Explore Pydantic validation

Create an invalid response dictionary. For example, remove one of the required fields from a valid Ollama response:

```python
invalid_response = {
    "model": "llama3.2:1b",
    "response": "Hello!"
    # "done" is missing
}
```

Try to validate it using `OllamaResponse.model_validate(...)`.

1. What exception does Pydantic raise?
2. What information does the exception give you about the invalid data?

### 2a. Create the APIClient class

Fill in the base class below. `_request` should be the *only* place in your
entire notebook that calls `requests.get` / `requests.post` and checks
`status_code`.

In [94]:
class APIClient:
    """Base class for talking to a JSON HTTP API."""

    def __init__(self, base_url: str):
        self.base_url = base_url

    def _request(self, method: str, endpoint: str, **kwargs) -> dict:
        url = f"{self.base_url}{endpoint}"
        response = requests.request(method, url, **kwargs)

        if response.status_code != 200:
            raise RuntimeError(
                f"Request to {url} failed with status {response.status_code}: "
                f"{response.text}"
            )

        return response.json()

    def format_prompt(self, *args, **kwargs) -> str:
        raise NotImplementedError


### 2b. Create the `OllamaClient` subclass

The following cell replaces the standalone `OllamaClient` from Exercise 1.
It now inherits the shared request logic from `APIClient`.

Implement an `OllamaClient` subclass that wraps the Ollama API at `http://localhost:11434`.

The class should:

1. Inherit from `APIClient`.
2. Use `super().__init__(...)` to configure the Ollama `base_url`.
3. Store the name of the Ollama model to use.
4. Implement a `generate(prompt)` method that:
   - Receives a prompt as a string.
   - Creates the appropriate JSON payload with `model`, `prompt`, and `stream`.
   - Calls `self._request(...)` to send the request.
   - Validates the response using the `OllamaResponse` Pydantic model created in Exercise 1.
   - Returns the validated `OllamaResponse`.

Finally, instantiate `OllamaClient`, send a prompt of your choice, and print the generated response.

In [110]:
class OllamaClient(APIClient):
    def __init__(self, model: str):
        super().__init__(base_url = "http://localhost:11434")
        self.model = model

    def generate(self, prompt: str) -> OllamaResponse:
        payload = {"model": self.model, "prompt": prompt, "stream": False}
        data = self._request("POST", "/api/generate", json = payload)
        return OllamaResponse.model_validate(data)

client = OllamaClient("llama3.2:1b")
result = client.generate("Explain inheritance in one sentence.")
print(result.response)


Inheritance is a fundamental concept in programming and object-oriented programming, where a class (the inheritor) inherits the properties and behaviors of another class (the base class) and can also add new properties and behaviors or override the ones inherited from the base class.


> #### 📚 Further reading
> - Nelson, *Software Engineering for Data Scientists*, **Chapter 4 "Object-Oriented Programming and Functional Programming"** — the foundational chapter for classes, methods, attributes, and defining your own classes. Read this first if `super().__init__()` above still feels shaky.
> - Nelson, **Chapter 8 "Design and Refactoring"** — covers modular code, interfaces and contracts, and coupling. This is the chapter that argues *why* you refactor procedural scripts into classes in the first place.
> - Ramalho, *Fluent Python* (2nd ed.), **Chapter 14 "Inheritance: For Better or for Worse"** — covers `super()` in more depth, plus multiple inheritance and method resolution order. Ramalho is more skeptical of inheritance than Nelson is, and argues for favoring composition in many cases. Worth reading side by side with Nelson's Chapter 4. Also check the wrap-up question at the end of this notebook.

---
## Exercise 3: Handling Validation Errors

Our `OllamaClient.generate()` method now converts the API response into an `OllamaResponse` Pydantic object.

But what happens if the API returns data that does **not** match the structure we expect?

### 3a. Explore Pydantic validation

Create an invalid response dictionary. For example, remove one of the required fields from a valid Ollama response:

```python
invalid_response = {
    "model": "llama3.2:1b",
    "response": "Hello!"
    # "done" is missing
}
```

Try to validate it using `OllamaResponse.model_validate(...)`.

1. What exception does Pydantic raise?
2. What information does the exception give you about the invalid data?

In [ ]:
class OllamaClient(APIClient):
    def __init__(self, model: str):
        super().__init__(base_url = "http://localhost:11434")
        self.model = model

    def generate(self, prompt: str) -> OllamaResponse:
        payload = {"model": self.model, "prompt": prompt, "stream": False}
        data = self._request("POST", "/api/generate", json = payload)
        
        try:
            return OllamaResponse.model_validate(data)
        except ValidationError as e:
            raise ValueError(
                f"Ollama response does not match expected output: {e}"
            )
        
    

client = OllamaClient("llama3.2:1b")
result = client.generate("Explain inheritance in one sentence.")
print(result.response)


### 3b. Handle validation errors in `generate`

Modify the generate() method of your existing OllamaClient class so that it:

1. Sends the request using `self._request(...)`.
2. Validates the returned JSON using `OllamaResponse`.
3. Catches `ValidationError` if the response does not match the expected structure.
4. Raises a clearer error explaining that the Ollama response could not be validated.

Test your error handling using the invalid response from the previous exercise.

In [112]:
from pydantic import ValidationError

invalid_response = {
    "model": "llama3.2:1b",
    "response": "Hello!"
    # "done" is missing
}

try:
    OllamaResponse.model_validate(invalid_response)
except ValidationError as exc:
    print(exc)

1 validation error for OllamaResponse
done
  Field required [type=missing, input_value={'model': 'llama3.2:1b', 'response': 'Hello!'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [113]:
class OllamaClient(APIClient):
    # TODO: implement the inherited client and handle validation errors.
    pass


### 3c. Put it together end to end

Using `OllamaClient`, reproduce the Ollama interaction from the previous lab in just a few lines:

1. Create an instance of `OllamaClient`.
2. Call `generate(...)` with a prompt of your choice.
3. Print the generated text using `.response`.

Compare this with the procedural version from the previous lab, where you had to manually:

- Define the URL and payload.
- Send the HTTP request.
- Check the status code.
- Parse the JSON response.
- Extract the generated text.

What responsibilities are now hidden behind the `OllamaClient` interface?

In [114]:
# TODO: instantiate OllamaClient and make an end-to-end request.


> #### 📚 Further reading
> - Nelson, *Software Engineering for Data Scientists*, **Chapter 11 "APIs"** — covers calling an API, HTTP methods and status codes, and building your own API with FastAPI. Good confirmatory reading for the request-handling logic inside `_request`, and a pointer to what "the other side" (building rather than consuming an API) looks like.

---
### Optional Stretch: ChatSession

Refactor Lab 2 Exercise 3 chat loop into a class that wraps an `OllamaClient`. Import any missing packages.

Design goals:
- `messages` should become an **instance attribute**, and not a loose global list.
- A `send(user_input: str) -> str` method appends the user message, calls the
  model, appends and returns the assistant's reply.
- The `while True` input loop becomes a `run()` method that calls
  `send` and prints the reply.


In [115]:
import ollama

class ChatSession:
    # TODO: initialize the client and message history, then implement the chat loop.
    pass


---

## Wrap-up questions (answer briefly in markdown)

**1. Which functionality is most appropriate to keep in the base `APIClient` class?**

A) Building Ollama prompts and choosing which model to use  
B) Sending HTTP requests, checking status codes, and parsing JSON responses  
C) Defining the fields of `OllamaResponse`  
D) Deciding what prompt the user should send  


**2. What is the main advantage of validating an API response with Pydantic?**

A) It makes the HTTP request faster  
B) It automatically fixes errors returned by the API  
C) It checks that the response has the expected structure and types as soon as
it is received  
D) It converts every API response into a string  


**3. Suppose we want to add a client for another JSON-based API. What would be the most appropriate approach with our current design?**

A) Add all the new API-specific methods directly to `APIClient`  
B) Copy the complete `OllamaClient` class and change its URL  
C) Create another subclass of `APIClient` containing the behavior specific to
the new API  
D) Modify `_request()` every time we add a new API  


**4. What does `super().__init__(...)` do inside `OllamaClient`?**

A) Creates a second `OllamaClient` object  
B) Calls the parent class's `__init__` method  
C) Sends a request to the API  
D) Automatically initializes every attribute in `OllamaClient`  


**5. Why does `OllamaClient` inherit from `APIClient`?**

A) Because `OllamaClient` needs to reuse general API request functionality  
B) Because Pydantic requires all API clients to use inheritance  
C) Because `requests.post()` only works inside a parent class  
D) Because every Python class must inherit from a custom class  


**6. Which statement best describes the inheritance vs. composition trade-off for this example?**

A) Inheritance is always better because it requires less code  
B) Composition is always better because inheritance should never be used  
C) Inheritance is reasonable because `OllamaClient` is a specialized API
client, while composition could provide more flexibility for replacing
components such as the request handler  
D) There is no meaningful difference between inheritance and composition  
